<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

# L2: Deep Agents, A Primer

<img src="images/DeepAgentIntro.png" alt="Deep Agents" width="350">

<!-- INTRODUCTION: your text goes here. -->

This primer covers two things a deep agent gives you that a plain tool-calling
agent does not: a filesystem, and instructions loaded from files. Each section
builds the smallest agent that shows the feature working.

In [1]:
# Setup and imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

from IPython.display import display, Markdown
from deepagent_helpers import langsmith_status
from visualization_helpers import render_report
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.messages import HumanMessage


load_dotenv(override=True)

MODEL = "openai:gpt-5.4-mini"
ROOT = "primer_workspace"      # the agent's filesystem root


def clear_poems():
    """Remove poems from an earlier run, so the output below is from this one.    """
    for poem in Path(ROOT, "poems").glob("*.md"):
        poem.unlink()


clear_poems()

print("✓ Environment ready")

✓ Environment ready


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

## The Filesystem

A deep agent gets file tools by default: `read_file`, `write_file`, `edit_file`,
`ls`, `glob`, `grep`, and `delete`. deepagents supports a variety of backends
including persistent databases and the local filesystem. For this course you
will use the local filesystem.

`FilesystemBackend(root_dir=..., virtual_mode=True)` is what points those tools
at a directory. The agent sees `root_dir` as `/`, and paths above it are
unreachable, so `/skills/haiku/SKILL.md` means
`primer_workspace/skills/haiku/SKILL.md`, and nothing outside that directory is
in range.

In [2]:
backend = FilesystemBackend(root_dir=ROOT, virtual_mode=True)

print("\nwhat the agent sees as its filesystem:")
for f in sorted(Path(ROOT).rglob("*")):
    if f.is_file():
        print("   /" + str(f.relative_to(ROOT)))


what the agent sees as its filesystem:
   /memories/AGENTS.md
   /skills/acrostic/SKILL.md
   /skills/haiku/SKILL.md
   /skills/limerick/SKILL.md
   /skills/sonnet/SKILL.md


## Instructions: AGENTS.md and Skills

Instructions can reach the agent three ways, a system prompt, an AGENTS.md file and SKILL.md files. The difference is where they are specified and when they load.

**`system_prompt`** is specified when the agent is created and used every time the agent is called.

**`memory=["/memories/AGENTS.md"]`** loads that file into the prompt/context at
the start of every run. Use it for what is always true: the agent's role, the
procedure it follows, the rules it must not break. It costs context on every
call, so it should stay short. In this lesson, it specifies the agent is a family-friendly poet.

**`skills=["/skills/"]`** works by *progressive disclosure*. Each skill is a
directory holding a `SKILL.md` with YAML frontmatter:

```
---
name: haiku
description: Write a haiku, a three line poem with a 5-7-5 syllable pattern...
---

(the full instructions follow)
```

Only the `name` and `description` are loaded up front. The agent reads the body
with `read_file` when it decides a skill applies. So twenty skills cost twenty
lines of context, not twenty documents, and the agent pulls in the detail for
the one it needs.

Our poet has one AGENTS.md and four skills:

```
primer_workspace/
  memories/AGENTS.md          "You are a poet that has skills..."
  skills/haiku/SKILL.md
  skills/limerick/SKILL.md
  skills/sonnet/SKILL.md
  skills/acrostic/SKILL.md
  poems/                      output
```

This layout may look familiar. Claude Code, Cursor, and other coding assistants
organize instructions the same way, for the same reason: the always-on rules go
in one file, and the specialized procedures sit behind a description the agent
reads only when it matters.

In [3]:
poet = create_deep_agent(
    model=MODEL,
    system_prompt="Your instructions are in /memories/AGENTS.md.",
    backend=backend,
    memory=["/memories/AGENTS.md"],
    skills=["/skills/"],
    middleware=[ModelCallLimitMiddleware(run_limit=30, exit_behavior="end")],
)
print("\u2713 Poet agent built")

✓ Poet agent built


## Optional: Tracing with LangSmith

The trace table below is built by hand from the agent's own stream. [LangSmith](https://smith.langchain.com) captures the same thing automatically for every run: every Model call, tool call, and retry, with inputs, outputs, latency, and tokens.

**No code change required.** Set three environment variables in `.env`:

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=your-api-key-here
LANGSMITH_PROJECT=loop-engineering-lesson-2
```

In [4]:
print(langsmith_status())

✓ LangSmith tracing is ON — after a live run, go to https://smith.langchain.com, open Tracing in the left-hand menu, and select the project 'AgentLoop'.


In [5]:
request = HumanMessage(content=(
    "Write a poem about the Boston Red Sox in each of your forms: haiku, "
    "limerick, sonnet, and an acrostic on the word BOSTON. Write each poem to "
    "its own file under /poems/, named after the form."
))

result = poet.invoke(
    {"messages": [request]},
    config={"run_name": "primer / poet, four forms",
            "tags": ["deepagents-primer", "skills"]},
)

# The read_file calls on SKILL.md are progressive disclosure happening: the
# agent pulled in each form's rules only when it came to write that form.
print("tool calls, in order:\n")
for m in result["messages"]:
    for c in (getattr(m, "tool_calls", None) or []):
        print(f"  {c['name']:12} {c['args'].get('file_path', '')}")

for poem in sorted(Path(ROOT, "poems").glob("*.md")):
    # Two trailing spaces force a Markdown line break: a poem has no blank
    # lines between verses, so without this every line would collapse into
    # one run-on paragraph.
    text = poem.read_text().strip().replace("\n", "  \n")
    display(render_report(text, title=f"/poems/{poem.name}"))

print("\n" + langsmith_status())

tool calls, in order:

  read_file    /skills/haiku/SKILL.md
  read_file    /skills/limerick/SKILL.md
  read_file    /skills/sonnet/SKILL.md
  read_file    /skills/acrostic/SKILL.md
  write_file   /poems/haiku.md
  write_file   /poems/limerick.md
  write_file   /poems/sonnet.md
  write_file   /poems/acrostic.md



✓ LangSmith tracing is ON — after a live run, go to https://smith.langchain.com, open Tracing in the left-hand menu, and select the project 'AgentLoop'.


### The trace

https://smith.langchain.com/public/64ca4edf-5bbb-4193-921d-3dc20b7bb9eb/r/01a068a7-054d-7fc0-9ce8-c9eb9d10638d?start_time=2026-09-03T19%3A02%3A51.469959Z

The LangSmith trace shows the system prompt with AGENTS.md and the four skill
descriptions in it, then each `read_file` on a `SKILL.md` arriving mid-run as
the agent decides it needs that form's rules.

Note, you do not need to sign up for LangSmith to view the traces that are linked to in this course.

## Conclusion

Two features, each one line of configuration:

- **A filesystem.** `backend=FilesystemBackend(root_dir=..., virtual_mode=True)`
  gives the agent file tools scoped to one directory, and makes its output
  something you can open rather than something you have to parse out of a
  message.
- **Instructions as files.** `memory=` loads what is always true on every run.
  `skills=` publishes a description per skill and lets the agent read the full
  procedure only when it applies, so a large library stays cheap.

There are many more features in Deep Agents. To discover more, you can check out a Deep Agent course [here](https://academy.langchain.com/courses/foundation-introduction-to-deepagents)